## 语义搜索轨迹

计算两两相邻的语义距离序列，并分析在调用AI事件前后，语义距离的变化情况。

In [ ]:
# 通用库
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

### 一、计算两两相邻的语义距离序列

对于某一个被试在某一个试次的$n$个想法序列$(I_1, I_2, \ldots, I_n)$，其两两相邻的语义距离序列$D=(d(I_1, I_2), d(I_2, I_3), \ldots, d(I_{n-1}, I_n))$，其中$d(I_i, I_j)$表示想法$I_i$和$I_j$之间的语义距离。

In [ ]:
def build_semantic_step_dataframe(df, ai_events_df=None):
    """
    基于单个试次的用户输入记录构建"语义步长表" (Semantic Step DataFrame)。
    
    Args:
        df (pd.DataFrame): 某一个被试在某一个试次的想法记录 (例如 user_msgs 中过滤出来的数据)。必须按时间(time)升序排列。包含 'content', 'time' 等列。
        ai_events_df (pd.DataFrame, optional): AI 调用事件表，用于标记每个语义步长的时间窗口内是否有 AI 调用发生。
        
    Returns:
        pd.DataFrame: 包含两两相邻想法间的语义步长信息 (思考时长、语义距离等)的 DataFrame。
    """
    if df is None or len(df) < 2:
        return pd.DataFrame()
        
    # 构建当前想法到下一个想法的映射
    step_df = df.copy()
    step_df['target_content'] = step_df['content'].shift(-1)
    step_df['target_time'] = step_df['time'].shift(-1)
    
    # 丢弃最后一条数据，因为它没有下一个想法作为目标
    step_df = step_df.dropna(subset=['target_content']).copy()
    
    # 计算完成这个语义步长所花费的发散思考时长
    step_df['time_span'] = step_df['target_time'] - step_df['time']
    
    # 动态将相对路径加入 sys.path
    toolbox_path = str(Path('../../../modules/semantic_toolbox').resolve())
    if toolbox_path not in sys.path:
        sys.path.insert(0, toolbox_path)
        
    # 动态导入模块，避免全局导入卡死
    from semdistance.distance import dis_sentences
    
    # 计算相邻两步之间的语义距离（即语义步长）
    def calc_distance(row):
        source = row['content']
        target = row['target_content']
        if pd.isna(source) or pd.isna(target):
            return np.nan
        try:
            return dis_sentences(source, target, 1)
        except Exception as e:
            print(f"计算出错: '{source}' vs '{target}', Error: {e}")
            return np.nan
            
    step_df['semantic_distance'] = step_df.apply(calc_distance, axis=1)
    
    # 标记每个语义步长时间窗口内是否有 AI 调用
    if ai_events_df is not None and len(ai_events_df) > 0:
        pid = df['participant_id'].iloc[0]
        item = df['item_name'].iloc[0]
        trial = df['trial_index'].iloc[0]
        trial_ai = ai_events_df[
            (ai_events_df['participant_id'] == pid) &
            (ai_events_df['item_name'] == item) &
            (ai_events_df['trial_index'] == trial)
        ]
        ai_times = trial_ai['click_time'].values
        step_df['has_ai_call'] = step_df.apply(
            lambda row: 1 if np.any((ai_times >= row['time']) & (ai_times <= row['target_time'])) else 0,
            axis=1
        )
    else:
        step_df['has_ai_call'] = 0
    
    # 标记这是一轮试次中的第几个步长
    step_df['step_index'] = range(1, len(step_df) + 1)
    
    # 重命名列以增加可读性：明确这是 source -> target
    step_df = step_df.rename(columns={
        'content': 'source_content',
        'time': 'source_time'
    })
    
    # 保留以下核心列，并保证它们的顺序
    base_cols = ['participant_id', 'item_name', 'trial_index', 'step_index', 
                 'source_content', 'target_content', 'source_time', 'target_time', 
                 'time_span', 'semantic_distance', 'has_ai_call']

    available_cols = [c for c in base_cols if c in step_df.columns]
    
    return step_df[available_cols]

In [ ]:
def calculate_all_semantic_steps():
    """
    读取所有用户的想法记录，计算所有试次的语义步长序列并合并为一张总表。
    """
    # 读取用户想法数据
    data_path = Path('../../../data/pickle/user_msgs.pkl')
    if not data_path.exists():
        print(f"找不到数据文件: {data_path}")
        return pd.DataFrame()
        
    user_msgs = pd.read_pickle(data_path)
    
    # 读取 AI 事件数据（用于标记语义步长窗口内是否有 AI 调用）
    ai_events_path = Path('../../../data/pickle/ai_events.pkl')
    ai_events_df = pd.read_pickle(ai_events_path) if ai_events_path.exists() else None
    
    all_steps = []
    
    # 按照被试和试次分组
    grouped = user_msgs.groupby(['participant_id', 'item_name', 'trial_index'])
    
    # 开始计算每个试次的语义步长序列
    for _, group in tqdm(grouped, desc="计算语义步长(按试次)"):
        # 必须保证想法是按时间严格生成先后排列的
        group = group.sort_values('time')
        
        # 调用此前构建好的单试次计算函数
        step_df = build_semantic_step_dataframe(group, ai_events_df)
        
        if not step_df.empty:
            all_steps.append(step_df)
            
    # 合成总表
    if all_steps:
        final_steps_df = pd.concat(all_steps, ignore_index=True)
        return final_steps_df
        
    return pd.DataFrame()

In [ ]:
# 首先从本地读取数据，如果不存在再执行计算
output_path = Path('output/semantic_steps_by_trial.csv')
# 强制重刷标识
force_recompute = False

if output_path.exists() and not force_recompute:
    print(f"从本地文件加载语义步长数据: {output_path}")
    all_steps_df = pd.read_csv(output_path)
else:
    # 执行计算并展示结果
    all_steps_df = calculate_all_semantic_steps()

# 展示前几行结果
display(all_steps_df.head())

# 保存结果到本地
all_steps_df.to_csv(output_path, index=False)

In [ ]:
# 计算累积语义距离（当前想法与试次内之前所有想法累积集合的语义距离）

def build_cumulative_semantic_dataframe(df):
    """
    计算每个想法与之前所有想法内容拼接后的语义距离。
    """
    if df is None or len(df) < 1:
        return pd.DataFrame()
        
    cum_df = df.copy()
    
    toolbox_path = str(Path('../../../modules/semantic_toolbox').resolve())
    if toolbox_path not in sys.path:
        sys.path.insert(0, toolbox_path)
        
    from semdistance.distance import dis_sentences
    
    cum_distances = []
    for i in range(len(cum_df)):
        if i == 0:
            cum_distances.append(np.nan) # 第一个想法之前没有历史文本
        else:
            # 将该想法之前的所有想法拼接为当前所在语境
            past_text = " ".join(cum_df['content'].iloc[:i].astype(str))
            current_text = str(cum_df['content'].iloc[i])
            try:
                # 计算当前想法与已有整体语境的语义距离
                d = dis_sentences(past_text, current_text, 1)
            except Exception as e:
                d = np.nan
            cum_distances.append(d)
            
    cum_df['cumulative_semantic_distance'] = cum_distances
    
    base_cols = ['participant_id', 'item_name', 'trial_index', 'time', 'content', 'cumulative_semantic_distance']
    available_cols = [c for c in base_cols if c in cum_df.columns]
    
    return cum_df[available_cols]

def calculate_all_cumulative_semantics():
    data_path = Path('../../../data/pickle/user_msgs.pkl')
    if not data_path.exists():
        print(f"找不到数据文件: {data_path}")
        return pd.DataFrame()
        
    user_msgs = pd.read_pickle(data_path)
    
    all_cum = []
    grouped = user_msgs.groupby(['participant_id', 'item_name', 'trial_index'])
    
    for _, group in tqdm(grouped, desc="计算累积语义距离"):
        group = group.sort_values('time')
        res = build_cumulative_semantic_dataframe(group)
        if not res.empty:
            all_cum.append(res)
            
    if all_cum:
        return pd.concat(all_cum, ignore_index=True)
    return pd.DataFrame()

# 从本地读取或重新计算
cum_output_path = Path('output/cumulative_semantic_distance.csv')
force_recompute_cum = False

if cum_output_path.exists() and not force_recompute_cum:
    print(f"从本地文件加载累积语义数据: {cum_output_path}")
    cum_df = pd.read_csv(cum_output_path)
else:
    cum_df = calculate_all_cumulative_semantics()

display(cum_df.head())
cum_df.to_csv(cum_output_path, index=False)

### 

### 二、可视化

In [ ]:
import matplotlib.pyplot as plt

# 解决中文显示问题
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 阶梯图绘图函数
def plot_semantic_trace(participant_id, item_name=None, trial_index=None,
                         steps_df=None, ai_events_df=None, originality_df=None,
                         show_labels=True):
    """
    绘制单个试次的语义搜索轨迹（时间轴阶梯图 + AI 调用标记 + 原创性）。

    x 轴为实验时间（0~300s），左侧 y 轴为语义距离，右侧 y 轴为原创性分数。
    每个想法按其提交时间定位，两两之间的台阶宽度等于实际时间间隔。

    Parameters
    ----------
    participant_id : int
    item_name : str, optional
        物品名称。若提供则优先使用（忽略 trial_index）。
    trial_index : int, optional
        试次编号。仅在 item_name 为 None 时生效。
    steps_df : pd.DataFrame
        all_steps_df，包含所有语义步长数据。
    ai_events_df : pd.DataFrame, optional
        AI 调用事件表，用于叠加 AI 调用标记。
    originality_df : pd.DataFrame, optional
        原创性评分表，用于在右侧 y 轴叠加每个想法的原创性散点。
    show_labels : bool
        是否显示想法文字标签。默认 True。
    """
    if steps_df is None:
        raise ValueError("请提供 steps_df 参数（语义步长数据表）")
    if item_name is None and trial_index is None:
        raise ValueError("请提供 item_name 或 trial_index 以确定试次")

    # —— 过滤出目标试次 ——
    mask = steps_df['participant_id'] == participant_id
    if item_name is not None:
        mask &= (steps_df['item_name'] == item_name)
        trial_label = item_name
    else:
        mask &= (steps_df['trial_index'] == trial_index)
        trial_label = f'trial {trial_index}'
    trial = steps_df.loc[mask].sort_values('step_index').copy()

    if trial.empty:
        desc = item_name if item_name is not None else f'trial {trial_index}'
        print(f"未找到被试 {participant_id}，{desc} 的数据")
        return

    # 时间轴 x 值：每个想法的提交时间
    source_times = trial['source_time'].values
    last_time = trial['target_time'].values[-1]
    x_steps = np.append(source_times, last_time)

    # y 值（语义距离），最后一段延续上一个值以补全阶梯
    y = trial['semantic_distance'].values
    y_steps = np.append(y, y[-1])

    idea_times = x_steps

    # —— 创建图表 ——
    fig, ax = plt.subplots(figsize=(12, 5))

    # 阶梯图：where='post' 使 y[i] 在区间 [x[i], x[i+1]) 上保持不变
    ax.step(x_steps, y_steps, where='post', linewidth=2, color='#2E86AB', label='语义步长')

    # 在每个台阶中点上方标注距离值
    for i in range(len(y)):
        mid_x = (x_steps[i] + x_steps[i+1]) / 2
        ax.text(mid_x, y[i] + 0.02, f'{y[i]:.3f}', ha='center', va='bottom',
                fontsize=7, color='#2E86AB')

    # 每个想法的提交时间用竖线标记
    IDEA_COLOR = '#2E86AB'
    for t in idea_times:
        ax.axvline(x=t, color=IDEA_COLOR, linewidth=1.2, alpha=0.55, zorder=0)
    for i, t in enumerate(idea_times):
        ax.text(t + 2, -0.05, str(i + 1), ha='center', va='top', fontsize=7, color='#333333')
    ax.plot([], [], color=IDEA_COLOR, linewidth=1.2, label='想法')

    # —— AI 调用标记 ——
    AI_COLOR = '#D64045'
    if ai_events_df is not None:
        ai_mask = (ai_events_df['participant_id'] == participant_id)
        if item_name is not None:
            ai_mask &= (ai_events_df['item_name'] == item_name)
        else:
            ai_mask &= (ai_events_df['trial_index'] == trial_index)
        ai_trial = ai_events_df[ai_mask].sort_values('click_time')

        for _, ai in ai_trial.iterrows():
            ct = ai['click_time']
            ax.axvline(x=ct, color=AI_COLOR, linestyle=(0, (4, 2)), linewidth=1.2, alpha=0.55)

        ax.plot([], [], color=AI_COLOR, linestyle=(0, (4, 2)), linewidth=1.2, label='AI 调用')

    # —— 原创性标记（右侧 y 轴）——
    ORIG_COLOR = '#E67E22'
    if originality_df is not None:
        orig_mask = (originality_df['participant_id'] == participant_id) & (originality_df['role'] == 'user')
        if item_name is not None:
            orig_mask &= (originality_df['item_name'] == item_name)
        else:
            orig_mask &= (originality_df['trial_index'] == trial_index)
        orig_trial = originality_df[orig_mask]

        if not orig_trial.empty:
            orig_map = dict(zip(orig_trial['time'].round(4), orig_trial['originality']))
            idea_orig = [orig_map.get(round(t, 4), np.nan) for t in idea_times]

            ax2 = ax.twinx()
            ax2.set_ylabel('原创性', fontsize=10)
            ax2.scatter(idea_times, idea_orig, color=ORIG_COLOR, s=36, zorder=10,
                        edgecolors='white', linewidth=0.5)

            # 右侧 y 轴范围
            orig_valid = [v for v in idea_orig if not np.isnan(v)]
            if orig_valid:
                o_min, o_max = min(orig_valid), max(orig_valid)
                o_pad = (o_max - o_min) * 0.15 if o_max > o_min else 0.2
                ax2.set_ylim(o_min - o_pad, o_max + o_pad)

        ax.plot([], [], 'o', color=ORIG_COLOR, markersize=6,
                markeredgecolor='white', markeredgewidth=0.5, label='原创性')

    # —— 坐标轴与标题 ——
    ax.set_xlim(0, 300)
    ax.set_xlabel('试次时间 (s)', fontsize=10)
    ax.set_ylabel('语义距离', fontsize=10)

    avg_d = y.mean()
    ax.axhline(y=avg_d, color='#888888', linestyle=':', linewidth=0.8, alpha=0.5)
    ax.text(298, avg_d + 0.01, f'M={avg_d:.3f}', ha='right', fontsize=7, color='#888888')

    ax.set_title(f'语义搜索轨迹 — 被试 {participant_id} · {trial_label}',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=4)
    ax.set_ylim(bottom=-0.08, top=max(y.max() + 0.15, 1.0))

    # —— 在台阶上方标注来源想法文字 ——
    if show_labels:
        idea_contents = list(trial["source_content"].values) + [trial["target_content"].values[-1]]
        label_y = 0.10
        for i, t in enumerate(idea_times):
            label = str(idea_contents[i])[:12]
            ax.text(t, label_y, label, ha="center", fontsize=6.5,
                    color="#666666", style="italic", rotation=30)

    plt.tight_layout(rect=[0, 0.06, 1, 1])
    return fig

In [ ]:
# 加载 AI 事件数据（用于叠加 AI 调用标记）
ai_events_path = Path('../../../data/pickle/ai_events.pkl')
ai_events_df = pd.read_pickle(ai_events_path) if ai_events_path.exists() else None
print(f"AI 事件数据: {ai_events_df.shape if ai_events_df is not None else '未找到'}")

In [ ]:
# 加载原创性数据（用于右侧 y 轴叠加原创性散点）
originality_path = Path('../../../data/analysis/scoring/originality.csv')
originality_df = pd.read_csv(originality_path) if originality_path.exists() else None
print(f"原创性数据: {originality_df.shape if originality_df is not None else '未找到'}")

In [ ]:
PID = 1
ITEM = '刀'

plot_semantic_trace(
    participant_id=PID,
    item_name=ITEM,
    steps_df=all_steps_df,
    ai_events_df=ai_events_df,
    originality_df=originality_df,
    show_labels=True
)

### 三、RQ3分析

#### 有AI调用语义步长与整体语义步长的对比分析

In [ ]:
# 针对每个试次计算 有AI调用的平均语义步长，与该试次所有平均语义步长，然后得到2000个数据点。
def get_trial_semantic_step(trial_df):
    """
    对单个试次的语义步长表，计算两个指标：
    - ai_step_mean: 有 AI 调用的语义步长的平均 semantic_distance
    - trial_step_mean: 该试次所有语义步长的平均 semantic_distance
    
    Args:
        trial_df: 单个试次的语义步长 DataFrame（含 has_ai_call 列）
    
    Returns:
        dict: {'ai_step_mean': float or np.nan, 'trial_step_mean': float}
    """
    distances = trial_df['semantic_distance'].dropna()
    if len(distances) == 0:
        return {'ai_step_mean': np.nan, 'trial_step_mean': np.nan}
    
    ai_mask = trial_df['has_ai_call'] == 1
    ai_distances = trial_df.loc[ai_mask, 'semantic_distance'].dropna()
    
    return {
        'ai_step_mean': ai_distances.mean() if len(ai_distances) > 0 else np.nan,
        'trial_step_mean': distances.mean()
    }

In [ ]:
# 对所有试次计算上述指标，得到一个总表
def calculate_semantic_step_comparison(steps_df):
    """
    对所有试次逐个计算 AI 调用步长 vs 整体步长的对比指标。
    
    Args:
        steps_df: all_steps_df，包含所有试次的语义步长数据（含 has_ai_call 列）
    
    Returns:
        pd.DataFrame: 每个试次一行，含 participant_id, item_name, trial_index,
                      ai_step_mean, trial_step_mean, ai_step_count, total_step_count
    """
    grouped = steps_df.groupby(['participant_id', 'item_name', 'trial_index'])
    records = []
    for (pid, item, trial), group in grouped:
        metrics = get_trial_semantic_step(group)
        records.append({
            'participant_id': pid,
            'item_name': item,
            'trial_index': trial,
            'ai_step_mean': metrics['ai_step_mean'],
            'trial_step_mean': metrics['trial_step_mean'],
            'ai_step_count': int((group['has_ai_call'] == 1).sum()),
            'total_step_count': len(group)
        })
    return pd.DataFrame(records)

In [ ]:
# 计算对比指标总表
comparison_df = calculate_semantic_step_comparison(all_steps_df)

# 进行t-test，比较有AI调用的语义步长与整体语义步长是否存在显著差异
from scipy import stats

# 剔除没有 AI 调用的试次（ai_step_mean 为 NaN），只比较有 AI 调用的试次
valid = comparison_df.dropna(subset=['ai_step_mean'])

ai_group = valid['ai_step_mean']
trial_group = valid['trial_step_mean']

# 配对 t 检验：同一试次内，有 AI 步长 vs 整体步长
t_stat, p_value = stats.ttest_rel(ai_group, trial_group)

print(f"有效试次数（有AI调用）: {len(valid)}")
print(f"有AI调用的平均语义步长: M={ai_group.mean():.4f}, SD={ai_group.std():.4f}")
print(f"整体平均语义步长:       M={trial_group.mean():.4f}, SD={trial_group.std():.4f}")
print(f"配对t检验: t({len(valid)-1})={t_stat:.4f}, p={p_value:.4f}")
print(f"效应量 (Cohen's d): { (ai_group.mean() - trial_group.mean()) / trial_group.std():.4f}")

In [ ]:
# 补充分析：比较有AI调用的语义步长 vs 无AI调用的语义步长
# 从 comparison_df 重新计算，增加 non_ai_step_mean

def get_trial_semantic_step_with_nonai(trial_df):
    distances = trial_df['semantic_distance'].dropna()
    if len(distances) == 0:
        return {'ai_step_mean': np.nan, 'non_ai_step_mean': np.nan}
    
    ai_mask = trial_df['has_ai_call'] == 1
    non_ai_mask = trial_df['has_ai_call'] == 0
    
    ai_distances = trial_df.loc[ai_mask, 'semantic_distance'].dropna()
    non_ai_distances = trial_df.loc[non_ai_mask, 'semantic_distance'].dropna()
    
    return {
        'ai_step_mean': ai_distances.mean() if len(ai_distances) > 0 else np.nan,
        'non_ai_step_mean': non_ai_distances.mean() if len(non_ai_distances) > 0 else np.nan
    }

grouped = all_steps_df.groupby(['participant_id', 'item_name', 'trial_index'])
records = []
for (pid, item, trial), group in grouped:
    metrics = get_trial_semantic_step_with_nonai(group)
    records.append({
        'participant_id': pid,
        'item_name': item,
        'trial_index': trial,
        'ai_step_mean': metrics['ai_step_mean'],
        'non_ai_step_mean': metrics['non_ai_step_mean']
    })
nonai_comp_df = pd.DataFrame(records)

# 只保留同时有 AI 步长和非 AI 步长的试次
valid = nonai_comp_df.dropna(subset=['ai_step_mean', 'non_ai_step_mean'])

ai_group = valid['ai_step_mean']
non_ai_group = valid['non_ai_step_mean']

t_stat, p_value = stats.ttest_rel(ai_group, non_ai_group)

print(f"有效试次数（同时有AI和非AI步长）: {len(valid)}")
print(f"有AI调用的平均语义步长:   M={ai_group.mean():.4f}, SD={ai_group.std():.4f}")
print(f"无AI调用的平均语义步长:   M={non_ai_group.mean():.4f}, SD={non_ai_group.std():.4f}")
print(f"配对t检验: t({len(valid)-1})={t_stat:.4f}, p={p_value:.4f}")
print(f"效应量 (Cohen's d): { (ai_group.mean() - non_ai_group.mean()) / non_ai_group.std():.4f}")

#### 调用后原创性提升检验

In [ ]:
# 加载调用后指标数据
post_call_path = Path('../../../data/analysis/post_call/post_call_metrics.csv')
post_call_df = pd.read_csv(post_call_path)

# 先聚合到试次级：每个试次的 originality_boost 均值
trial_boost = post_call_df.groupby(
    ['participant_id', 'item_name', 'trial_index']
)['originality_boost'].mean().dropna()

n = len(trial_boost)
n_participants = trial_boost.index.get_level_values('participant_id').nunique()

print(f"有效试次数: {n}（来自 {n_participants} 名被试）")
print(f"试次级 originality_boost 描述统计:")
print(f"  M = {trial_boost.mean():.4f}")
print(f"  SD = {trial_boost.std():.4f}")
print(f"  Min = {trial_boost.min():.4f}")
print(f"  Max = {trial_boost.max():.4f}")
print(f"  正值比例 = {(trial_boost > 0).mean():.2%}")

# 单样本 t 检验（双尾 + 单尾）
t_stat, p_two_tailed = stats.ttest_1samp(trial_boost, 0)
p_one_tailed = p_two_tailed / 2 if t_stat > 0 else 1 - p_two_tailed / 2
cohens_d = trial_boost.mean() / trial_boost.std()

print(f"\n试次级单样本t检验 (H0: μ=0):")
print(f"  t({n-1}) = {t_stat:.4f}")
print(f"  p (双尾) = {p_two_tailed:.4f}")
print(f"  p (单尾, H1: μ>0) = {p_one_tailed:.4f}")
print(f"  Cohen's d = {cohens_d:.4f}")

# 可视化
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(trial_boost, bins=30, color='#2E86AB', alpha=0.7, edgecolor='white')
ax.axvline(0, color='#888888', linestyle='--', linewidth=1.2, label='零线 (无变化)')
ax.axvline(trial_boost.mean(), color='#D64045', linestyle='-', linewidth=1.5,
           label=f'均值 = {trial_boost.mean():.3f}')
ax.set_xlabel('试次级原创性提升均值', fontsize=11)
ax.set_ylabel('试次频次', fontsize=11)
ax.set_title(f'调用AI后原创性提升 — 试次级 (N={n})', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

#### 调用前语义距离衰减与调用频率

In [ ]:
# 7. 调用前语义距离衰减 → 调用频率
#   semantic_distance_change（调用前最后 3 条想法的语义距离变化趋势）与调用次数做相关。负值（搜索趋同）是否预测更高频的 AI 使用。

#### 机器学习

##### 预测是否调用

In [ ]:
# 构建回答级特征数据集
# 每个数据点 = 一个回答，特征描述"当前状态"，目标 = "在下一个回答出现前是否调用了AI"

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_val_predict
from sklearn.metrics import RocCurveDisplay
from sklearn.preprocessing import StandardScaler

# ——— 1. 从 all_steps_df 构建特征 ———
# all_steps_df 每行 = 从 idea_i (source) 到 idea_{i+1} (target) 的一步
# 特征来自 source 想法，目标是 has_ai_call
ml_df = all_steps_df.copy()
ml_df = ml_df.sort_values(['participant_id', 'item_name', 'trial_index', 'step_index'])

# 获取"到达该想法"的思考时间和语义步长（来自前一行）
ml_df['prev_time_span'] = ml_df.groupby(
    ['participant_id', 'item_name', 'trial_index']
)['time_span'].shift(1)

ml_df['prev_semantic_distance'] = ml_df.groupby(
    ['participant_id', 'item_name', 'trial_index']
)['semantic_distance'].shift(1)

# 第一个想法的思考时间 = 从试次开始到第一个想法的时间
first_mask = ml_df['step_index'] == 1
ml_df.loc[first_mask, 'prev_time_span'] = ml_df.loc[first_mask, 'source_time']
# 第一个想法没有前一个语义步长，后续会被剔除

# 构造特征
ml_df['think_time'] = ml_df['prev_time_span']
ml_df['semantic_step'] = ml_df['prev_semantic_distance']
ml_df['response_time'] = ml_df['source_time']

# ——— 2. 合并原创性 ———
orig_user = originality_df[originality_df['role'] == 'user'].copy()
orig_user['time_round'] = orig_user['time'].round(4)
ml_df['time_round'] = ml_df['source_time'].round(4)

ml_df = ml_df.merge(
    orig_user[['participant_id', 'item_name', 'trial_index', 'time_round', 'originality']],
    on=['participant_id', 'item_name', 'trial_index', 'time_round'],
    how='left'
)
ml_df = ml_df.drop(columns=['time_round'])

# ——— 3. 清理 ———
call_feature_cols = ['think_time', 'semantic_step', 'originality', 'response_time']
ml_df = ml_df.dropna(subset=call_feature_cols + ['has_ai_call'])

X = ml_df[call_feature_cols].values
call_y = ml_df['has_ai_call'].values.astype(int)

print(f"总回答数: {len(ml_df)}")
print(f"有AI调用的步长: {call_y.sum()} ({call_y.mean():.1%})")
print(f"无AI调用的步长: {(1 - call_y).sum()} ({(1 - call_y).mean():.1%})")
print(f"\n特征描述统计:")
display(ml_df[call_feature_cols].describe())

# ——— 4. 标准化 + 训练/评估 ———
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 逻辑回归
call_lr = LogisticRegression(C=1.0, random_state=42)
call_lr.fit(X_scaled, call_y)

call_lr_auc = cross_val_score(call_lr, X_scaled, call_y, cv=cv, scoring='roc_auc').mean()
call_lr_acc = cross_val_score(call_lr, X_scaled, call_y, cv=cv, scoring='accuracy').mean()

print(f"\n=== 逻辑回归 ===")
print(f"5-fold CV AUC: {call_lr_auc:.4f}")
print(f"5-fold CV Accuracy: {call_lr_acc:.4f}")
print(f"\n系数 (标准化后):")
for name, coef in zip(call_feature_cols, call_lr.coef_[0]):
    or_val = np.exp(coef)
    print(f"  {name:>16s}: B={coef:+.4f}, OR={or_val:.4f}")

# 随机森林
call_rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
call_rf.fit(X_scaled, call_y)

call_rf_auc = cross_val_score(call_rf, X_scaled, call_y, cv=cv, scoring='roc_auc').mean()
call_rf_acc = cross_val_score(call_rf, X_scaled, call_y, cv=cv, scoring='accuracy').mean()

print(f"\n=== 随机森林 ===")
print(f"5-fold CV AUC: {call_rf_auc:.4f}")
print(f"5-fold CV Accuracy: {call_rf_acc:.4f}")
print(f"\n特征重要性:")
for name, imp in sorted(zip(call_feature_cols, call_rf.feature_importances_), key=lambda x: -x[1]):
    print(f"  {name:>16s}: {imp:.4f}")

# ——— 5. 可视化 ———
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) 类别分布
ax = axes[0]
counts = [int((1 - call_y).sum()), int(call_y.sum())]
bars = ax.bar(['无AI调用', '有AI调用'], counts, color=['#2E86AB', '#D64045'], alpha=0.7, edgecolor='white')
ax.set_ylabel('回答数', fontsize=11)
ax.set_title(f'类别分布 (正类={call_y.mean():.1%})', fontsize=12, fontweight='bold')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, str(cnt),
            ha='center', fontsize=11)

# (b) ROC 曲线
ax = axes[1]
y_prob_lr = cross_val_predict(call_lr, X_scaled, call_y, cv=cv, method='decision_function')
RocCurveDisplay.from_predictions(call_y, y_prob_lr, ax=ax, color='#2E86AB', name=f'LR (AUC={call_lr_auc:.3f})')
y_prob_rf = cross_val_predict(call_rf, X_scaled, call_y, cv=cv, method='predict_proba')[:, 1]
RocCurveDisplay.from_predictions(call_y, y_prob_rf, ax=ax, color='#D64045', name=f'RF (AUC={call_rf_auc:.3f})')
ax.set_title('ROC 曲线 (5-fold CV)', fontsize=12, fontweight='bold')

# (c) 特征重要性对比
ax = axes[2]
x_pos = np.arange(len(call_feature_cols))
width = 0.35
lr_imp = np.abs(call_lr.coef_[0])
lr_imp = lr_imp / lr_imp.sum()
ax.bar(x_pos - width/2, lr_imp, width, color='#2E86AB', alpha=0.7, label='LR (|coef|归一化)')
ax.bar(x_pos + width/2, call_rf.feature_importances_, width, color='#D64045', alpha=0.7, label='RF (importance)')
ax.set_xticks(x_pos)
ax.set_xticklabels(['思考时间', '语义步长', '原创性', '回答时间'], fontsize=9)
ax.set_ylabel('相对重要性', fontsize=11)
ax.set_title('特征重要性对比', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

##### 预测是否有用

In [ ]:
# 数据点为一次 AI 调用，分为两类，有效调用和无效调用，前者的originality_boost>0，后者<=0
# 预测因子：调用前思考时间、调用前第一个想法的原创性（如有）、调用时间、调用前思考时间与平均回答思考时间的比值、调用前最后一个语义步长、调用前想法数(fluency)、调用前累积语义距离、近期流畅性、上一想法与提示词的语义距离

# ——— 1. 加载 pre_call 和 post_call 数据 ———
pre_path = Path('../../../data/analysis/pre_call/pre_call_metrics.csv')
post_path = Path('../../../data/analysis/post_call/post_call_metrics.csv')
pre_df = pd.read_csv(pre_path)
post_df = pd.read_csv(post_path)

# 合并：每行 = 一次 AI 调用，带调用前特征和调用后结果
call_df = pre_df.merge(
    post_df[['participant_id', 'item_name', 'trial_index', 'click_index', 'originality_boost']],
    on=['participant_id', 'item_name', 'trial_index', 'click_index'],
    how='inner'
)

# ——— 2. 获取调用前最后一条用户消息的原创性 ———
orig_user = originality_df[originality_df['role'] == 'user'].copy()
orig_user['time_round'] = orig_user['time'].round(4)
call_df['time_round'] = call_df['last_user_time'].round(4)

call_df = call_df.merge(
    orig_user[['participant_id', 'item_name', 'trial_index', 'time_round', 'originality']],
    on=['participant_id', 'item_name', 'trial_index', 'time_round'],
    how='left',
    suffixes=('', '_last')
)
call_df = call_df.drop(columns=['time_round'])

# ——— 2.5 获取调用前最后一个语义步长及调用前生成想法数 ———
def get_last_semantic_step(row):
    mask = (all_steps_df['participant_id'] == row['participant_id']) & \
           (all_steps_df['item_name'] == row['item_name']) & \
           (all_steps_df['trial_index'] == row['trial_index']) & \
           (all_steps_df['target_time'] < row['click_time'])
    steps = all_steps_df[mask]
    if not steps.empty: # 获取 target_time 最大的那个 step 的 semantic_distance
        return steps.sort_values('target_time', ascending=False).iloc[0]['semantic_distance']
    return np.nan

call_df['last_semantic_distance'] = call_df.apply(get_last_semantic_step, axis=1)

def get_pre_call_fluency(row):
    mask = (orig_user['participant_id'] == row['participant_id']) & \
           (orig_user['item_name'] == row['item_name']) & \
           (orig_user['trial_index'] == row['trial_index']) & \
           (orig_user['time'] < row['click_time'])
    return mask.sum()

call_df['pre_call_fluency'] = call_df.apply(get_pre_call_fluency, axis=1)

# ——— 2.6 获取调用前的累积语义距离 ———
cum_path = Path('output/cumulative_semantic_distance.csv')
if cum_path.exists():
    cum_df = pd.read_csv(cum_path)
    def get_last_cumulative(row):
        mask = (cum_df['participant_id'] == row['participant_id']) & \
               (cum_df['item_name'] == row['item_name']) & \
               (cum_df['trial_index'] == row['trial_index']) & \
               (cum_df['time'] < row['click_time'])
        past_ideas = cum_df[mask]
        if not past_ideas.empty:
            return past_ideas.sort_values('time', ascending=False).iloc[0]['cumulative_semantic_distance']
        return np.nan

    call_df['last_cumulative_distance'] = call_df.apply(get_last_cumulative, axis=1)
else:
    print("WARNING: cumulative_semantic_distance.csv not found. Please run the calculation cell first.")
    call_df['last_cumulative_distance'] = np.nan

# ——— 2.7 计算 local_fluency（近期流畅性：调用前1分钟内的产出数量） ———
def get_local_fluency(row):
    pid, item, trial = row['participant_id'], row['item_name'], row['trial_index']
    click_t = row['click_time']
    recent_mask = (orig_user['participant_id'] == pid) & \
                  (orig_user['item_name'] == item) & \
                  (orig_user['trial_index'] == trial) & \
                  (orig_user['time'] < click_t) & \
                  (orig_user['time'] >= click_t - 60)
    return recent_mask.sum()

call_df['local_fluency'] = call_df.apply(get_local_fluency, axis=1)

# ——— 2.8 计算 distance_to_prompt（上一想法与初始提示词的语义距离） ———
def get_distance_to_prompt(row):
    pid, item, trial = row['participant_id'], row['item_name'], row['trial_index']
    click_t = row['click_time']
    mask = (all_steps_df['participant_id'] == pid) & \
           (all_steps_df['item_name'] == item) & \
           (all_steps_df['trial_index'] == trial) & \
           (all_steps_df['target_time'] < click_t)
    steps = all_steps_df[mask]
    if steps.empty:
        return np.nan
    last_idea = steps.sort_values('target_time', ascending=False).iloc[0]['target_content']
    toolbox_path = str(Path('../../../modules/semantic_toolbox').resolve())
    if toolbox_path not in sys.path:
        sys.path.insert(0, toolbox_path)
    from semdistance.distance import dis_sentences
    try:
        return dis_sentences(last_idea, item, 1)
    except Exception:
        return np.nan

call_df['distance_to_prompt'] = call_df.apply(get_distance_to_prompt, axis=1)

# ——— 3. 构造特征和标签 ———
call_df['effective'] = (call_df['originality_boost'] > 0).astype(int)
# merge 后 originality 列名直接就是 'originality'，重命名为 'originality_last'
existing_cols = call_df.columns.tolist()
orig_col = 'originality_last' if 'originality_last' in existing_cols else 'originality'
if orig_col != 'originality_last':
    call_df = call_df.rename(columns={'originality': 'originality_last'})

feature_cols = ['pre_think_time', 'originality_last', 'click_time', 'relative_pause', 'last_semantic_distance', 'pre_call_fluency', 'last_cumulative_distance', 'local_fluency', 'distance_to_prompt']
call_df = call_df.dropna(subset=feature_cols + ['effective'])

X = call_df[feature_cols].values
y = call_df['effective'].values.astype(int)

from IPython.display import display
print(f"总 AI 调用次数: {len(call_df)}")
print(f"有效调用 (boost>0): {y.sum()} ({y.mean():.1%})")
print(f"无效调用 (boost<=0): {(1 - y).sum()} ({(1 - y).mean():.1%})")
print(f"\n特征描述统计:")
display(call_df[feature_cols].describe())

# ——— 4. 标准化 + 训练/评估 ———
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 逻辑回归
lr = LogisticRegression(C=1.0, random_state=42)
lr.fit(X_scaled, y)

lr_auc = cross_val_score(lr, X_scaled, y, cv=cv, scoring='roc_auc').mean()
lr_acc = cross_val_score(lr, X_scaled, y, cv=cv, scoring='accuracy').mean()

label_map = {
    'pre_think_time': '调用前思考时间',
    'originality_last': '调用前最后想法原创性',
    'click_time': '调用时间',
    'relative_pause': '思考时间/平均思考时间',
    'last_semantic_distance': '调用前语义步长',
    'pre_call_fluency': '调用前想法数',
    'last_cumulative_distance': '调用前累积距离',
    'local_fluency': '近期流畅性',
    'distance_to_prompt': '距提示词语义距离'
}

print(f"\n=== 逻辑回归 ===")
print(f"5-fold CV AUC: {lr_auc:.4f}")
print(f"5-fold CV Accuracy: {lr_acc:.4f}")
print(f"\n系数 (标准化后):")
for name, coef in zip(feature_cols, lr.coef_[0]):
    or_val = np.exp(coef)
    print(f"  {label_map[name]:>16s}: B={coef:+.4f}, OR={or_val:.4f}")

# 随机森林
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_scaled, y)

rf_auc = cross_val_score(rf, X_scaled, y, cv=cv, scoring='roc_auc').mean()
rf_acc = cross_val_score(rf, X_scaled, y, cv=cv, scoring='accuracy').mean()

print(f"\n=== 随机森林 ===")
print(f"5-fold CV AUC: {rf_auc:.4f}")
print(f"5-fold CV Accuracy: {rf_acc:.4f}")
print(f"\n特征重要性:")
for name, imp in sorted(zip(feature_cols, rf.feature_importances_), key=lambda x: -x[1]):
    print(f"  {label_map[name]:>16s}: {imp:.4f}")

# ——— 5. 可视化 ———
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) 类别分布
ax = axes[0]
counts = [int((1 - y).sum()), int(y.sum())]
bars = ax.bar(['无效调用', '有效调用'], counts, color=['#888888', '#D64045'], alpha=0.7, edgecolor='white')
ax.set_ylabel('调用次数', fontsize=11)
ax.set_title(f'类别分布 (有效={y.mean():.1%})', fontsize=12, fontweight='bold')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30, str(cnt),
            ha='center', fontsize=11)

# (b) ROC 曲线
ax = axes[1]
y_prob_lr = cross_val_predict(lr, X_scaled, y, cv=cv, method='decision_function')
RocCurveDisplay.from_predictions(y, y_prob_lr, ax=ax, color='#2E86AB', name=f'LR (AUC={lr_auc:.3f})')
y_prob_rf = cross_val_predict(rf, X_scaled, y, cv=cv, method='predict_proba')[:, 1]
RocCurveDisplay.from_predictions(y, y_prob_rf, ax=ax, color='#D64045', name=f'RF (AUC={rf_auc:.3f})')
ax.set_title('ROC 曲线 (5-fold CV)', fontsize=12, fontweight='bold')

# (c) 特征重要性对比
ax = axes[2]
x_pos = np.arange(len(feature_cols))
width = 0.35
lr_imp = np.abs(lr.coef_[0])
lr_imp = lr_imp / lr_imp.sum()
ax.bar(x_pos - width/2, lr_imp, width, color='#2E86AB', alpha=0.7, label='LR (|coef|归一化)')
ax.bar(x_pos + width/2, rf.feature_importances_, width, color='#D64045', alpha=0.7, label='RF (importance)')
ax.set_xticks(x_pos)
ax.set_xticklabels(['思考\n时间', '上想\n原创', '调用\n时间', '相对\n思考', '调用\n步长', '调用\n前数', '累积\n距离', '近期\n流畅', '距提\n示词'], fontsize=7)
ax.set_ylabel('相对重要性', fontsize=11)
ax.set_title('特征重要性对比', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# 树模型本身是对特征非线性及特征尺度不敏感的，但为了让横坐标展示真实的特征数值（而非经过标准化的值），
# 我们用原始数据 X 重新 fit 一个供解释用的随机森林模型
rf_unscaled = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_unscaled.fit(X, y)

# 选取随机森林中重要性最高的4个特征来观察非线性边缘效应
target_features = ['click_time', 'originality_last', 'distance_to_prompt', 'last_semantic_distance']
target_indices = [feature_cols.index(f) for f in target_features]
feature_names_cn = [label_map[f] for f in feature_cols]

fig, ax = plt.subplots(figsize=(18, 4.5))

pdp_display = PartialDependenceDisplay.from_estimator(
    rf_unscaled, X, features=target_indices,
    feature_names=feature_names_cn,
    n_cols=4, ax=ax, grid_resolution=50,
    line_kw={"color": "#D64045", "linewidth": 2}
)

fig.suptitle('部分依赖图 - 特征边际效应对预测概率的影响', fontsize=14, fontweight='bold', y=1.05)
plt.show()

In [ ]:
import json
from pathlib import Path
from scipy import stats

Path('output').mkdir(exist_ok=True)

# --- 1. semantic_step: AI vs overall ---
sem_valid = comparison_df.dropna(subset=['ai_step_mean'])
sem_ai = sem_valid['ai_step_mean']
sem_trial = sem_valid['trial_step_mean']
t_sem, p_sem = stats.ttest_rel(sem_ai, sem_trial)
d_sem = (sem_ai.mean() - sem_trial.mean()) / sem_trial.std()

# --- 2. originality_boost ---
t_boost, p_boost = stats.ttest_1samp(trial_boost, 0)
p_boost_one = p_boost / 2 if t_boost > 0 else 1 - p_boost / 2
d_boost = trial_boost.mean() / trial_boost.std()

r = {
    "semantic_step": {
        "n_trials": int(len(sem_valid)),
        "ai_step_M": round(float(sem_ai.mean()), 6),
        "ai_step_SD": round(float(sem_ai.std()), 6),
        "trial_step_M": round(float(sem_trial.mean()), 6),
        "trial_step_SD": round(float(sem_trial.std()), 6),
        "t_stat": round(float(t_sem), 4), "p_val": round(float(p_sem), 6),
        "cohens_d": round(float(d_sem), 6)
    },
    "originality_boost": {
        "n": int(len(trial_boost)), "M": round(float(trial_boost.mean()), 6),
        "SD": round(float(trial_boost.std()), 6),
        "positive_ratio": round(float((trial_boost > 0).mean()), 6),
        "t_stat": round(float(t_boost), 4), "p_two_tail": round(float(p_boost), 6),
        "p_one_tail": round(float(p_boost_one), 6), "cohens_d": round(float(d_boost), 6)
    },
    "ml_predict_call": {
        "n": int(len(ml_df)), "positive_rate": round(float(call_y.mean()), 6),
        "lr_auc": round(float(call_lr_auc), 4), "rf_auc": round(float(call_rf_auc), 4),
        "lr_coefs": [{"feature": fn, "B": round(float(c), 4), "OR": round(float(np.exp(c)), 4)}
                     for fn, c in zip(call_feature_cols, call_lr.coef_[0])],
        "rf_importance": [{"feature": fn, "importance": round(float(i), 4)}
                         for fn, i in sorted(zip(call_feature_cols, call_rf.feature_importances_), key=lambda x: -x[1])]
    },
    "ml_predict_effective": {
        "n": int(len(call_df)), "effective_rate": round(float(y.mean()), 6),
        "lr_auc": round(float(lr_auc), 4), "rf_auc": round(float(rf_auc), 4),
        "lr_coefs": [{"feature": label_map[f], "B": round(float(c), 4), "OR": round(float(np.exp(c)), 4)}
                     for f, c in zip(feature_cols, lr.coef_[0])],
        "rf_importance": [{"feature": label_map[f], "importance": round(float(i), 4)}
                         for f, i in sorted(zip(feature_cols, rf.feature_importances_), key=lambda x: -x[1])]
    }
}

Path('output/trial_semantic_trace.json').write_text(json.dumps(r, ensure_ascii=False, indent=2))
print("Exported trial_semantic_trace.json")